# MonoRace M23 PPO Training (Paper-Exact Replication)
**arxiv 2601.15222** — Full system-identified dynamics, SB3 PPO on CPU

Checkpoints saved to Google Drive so training persists across sessions.

In [ ]:
!pip install stable-baselines3 gymnasium tensorboard -q
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/drone2_training'
CHECKPOINT_DIR = os.path.join(SAVE_DIR, 'checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Saving to: {SAVE_DIR}")

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import gymnasium as gym
from gymnasium import spaces

# ═══════════════════════════════════════════════════════════════════
# M23 Configuration (Table 1, arxiv 2601.15222)
# ═══════════════════════════════════════════════════════════════════

GRAVITY = 9.81

# Motor parameters
MOTOR_OMEGA_MIN = 341.75
MOTOR_OMEGA_MAX = 3100.0
MOTOR_TAU = 0.025
MOTOR_K_CMD = 0.50
PROP_RADIUS = 0.0485775

# Force coefficients (specific accelerations)
K_OMEGA = 1.55e-6
K_X = 5.37e-5
K_Y = 5.37e-5
K_X2 = 4.10e-3
K_Y2 = 1.51e-2
K_ALPHA = 3.145
K_HOR = 7.245

# Per-motor moment coefficients
K_P = np.array([4.99e-5, 3.78e-5, 4.82e-5, 3.83e-5])
K_Q = np.array([2.05e-5, 2.46e-5, 2.02e-5, 2.57e-5])
K_R_SPEED = np.array([3.38e-3, 3.38e-3, 3.38e-3, 3.38e-3])
K_R_ACCEL = np.array([3.24e-4, 3.24e-4, 3.24e-4, 3.24e-4])
J_X, J_Y, J_Z = -0.89, 0.96, -0.34

# Domain randomization
DR_RANGE = 0.5
DR_OMEGA_MAX = 0.4
DR_TAU = 0.55

DT_SIM = 0.01  # 100 Hz

# Reward lambdas (M23)
LAMBDA_PROG = 1.0
LAMBDA_GATE = 1.5
LAMBDA_RATE = 0.001
LAMBDA_OFFSET = 1.5
LAMBDA_PERC = 0.01
LAMBDA_CRASH = 10.0
V_MAX = 10.0

OBS_DIM = 24

# Track layout (11 gates)
GATE_POSITIONS = [
    (5.0, -13.0, -2.0, 0.0),
    (15.0, -8.0, -2.5, 0.3),
    (25.0, -5.0, -2.0, 0.0),
    (30.0, -5.0, -2.0, 0.0),
    (42.0, -10.0, -3.5, -0.4),
    (55.0, -15.0, -4.0, np.pi),
    (50.0, -15.0, -1.5, 0.0),
    (62.0, -18.0, -2.0, -0.2),
    (75.0, -20.0, -2.5, 0.0),
    (80.0, -20.0, -2.5, 0.0),
    (92.0, -15.0, -2.0, 0.3),
]
GATE_SIZE = 0.40
NUM_LAPS = 2
NUM_GATES = len(GATE_POSITIONS)

# Init ranges
INIT_X_RANGE = (1.0, 95.0)
INIT_Y_RANGE = (-27.0, 1.0)
INIT_Z_RANGE = (-5.0, 0.0)
INIT_V_RANGE = (-0.5, 0.5)
INIT_RP_RANGE = (-np.pi / 9, np.pi / 9)
INIT_YAW_RANGE = (-np.pi, np.pi)
INIT_OMEGA_RANGE = (-0.1, 0.1)

# Bounds
BOUNDS_X = (1.0, 95.0)
BOUNDS_Y = (-27.0, 1.0)
BOUNDS_Z = (-6.0, 0.0)
OMEGA_MAX_TERMINATION = 29.6706  # 1700 deg/s

# Training
MAX_EPISODE_STEPS = 4000
NUM_ENVS = 8
TOTAL_TIMESTEPS = 50_000_000

print("Config loaded: M23 paper-exact parameters")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Quaternion / Euler utilities
# ═══════════════════════════════════════════════════════════════════

def quat_to_euler(q):
    """Quaternion [qw,qx,qy,qz] -> Euler [roll, pitch, yaw]."""
    qw, qx, qy, qz = q
    roll = np.arctan2(2 * (qw * qx + qy * qz), 1 - 2 * (qx**2 + qy**2))
    pitch = np.arcsin(np.clip(2 * (qw * qy - qz * qx), -1.0, 1.0))
    yaw = np.arctan2(2 * (qw * qz + qx * qy), 1 - 2 * (qy**2 + qz**2))
    return np.array([roll, pitch, yaw])

def euler_to_quat(roll, pitch, yaw):
    """Euler angles -> Quaternion [qw,qx,qy,qz]."""
    cr, sr = np.cos(roll/2), np.sin(roll/2)
    cp, sp = np.cos(pitch/2), np.sin(pitch/2)
    cy, sy = np.cos(yaw/2), np.sin(yaw/2)
    return np.array([
        cr*cp*cy + sr*sp*sy,
        sr*cp*cy - cr*sp*sy,
        cr*sp*cy + sr*cp*sy,
        cr*cp*sy - sr*sp*cy,
    ])

# ═══════════════════════════════════════════════════════════════════
# M23 System-Identified Dynamics (arxiv 2601.15222, Table 1)
# ═══════════════════════════════════════════════════════════════════

class QuadcopterDynamics(nn.Module):
    """Full M23 dynamics: aero drag, nonlinear motors, per-motor moments, gyroscopic coupling."""

    def __init__(self, dt=DT_SIM, params=None):
        super().__init__()
        p = params or {}
        self.dt = dt
        self.k_omega = p.get('k_omega', K_OMEGA)
        self.k_x = p.get('k_x', K_X)
        self.k_y = p.get('k_y', K_Y)
        self.k_x2 = p.get('k_x2', K_X2)
        self.k_y2 = p.get('k_y2', K_Y2)
        self.k_alpha = p.get('k_alpha', K_ALPHA)
        self.k_hor = p.get('k_hor', K_HOR)
        self.kp = torch.tensor(p.get('kp', K_P), dtype=torch.float32)
        self.kq = torch.tensor(p.get('kq', K_Q), dtype=torch.float32)
        self.kr_speed = torch.tensor(p.get('kr_speed', K_R_SPEED), dtype=torch.float32)
        self.kr_accel = torch.tensor(p.get('kr_accel', K_R_ACCEL), dtype=torch.float32)
        self.Jx = p.get('Jx', J_X)
        self.Jy = p.get('Jy', J_Y)
        self.Jz = p.get('Jz', J_Z)
        self.motor_tau = p.get('motor_tau', MOTOR_TAU)
        self.motor_k_cmd = p.get('motor_k_cmd', MOTOR_K_CMD)
        self.omega_min = p.get('omega_min', MOTOR_OMEGA_MIN)
        self.omega_max = p.get('omega_max', MOTOR_OMEGA_MAX)
        self.g = torch.tensor([0.0, 0.0, GRAVITY], dtype=torch.float32)
        self.prev_motor_speeds = None

    def _quat_to_rot_matrix(self, q):
        qw, qx, qy, qz = q[:, 0], q[:, 1], q[:, 2], q[:, 3]
        R = torch.empty((q.shape[0], 3, 3), device=q.device, dtype=q.dtype)
        R[:, 0, 0] = 1 - 2*(qy**2 + qz**2); R[:, 0, 1] = 2*(qx*qy - qw*qz); R[:, 0, 2] = 2*(qx*qz + qw*qy)
        R[:, 1, 0] = 2*(qx*qy + qw*qz); R[:, 1, 1] = 1 - 2*(qx**2 + qz**2); R[:, 1, 2] = 2*(qy*qz - qw*qx)
        R[:, 2, 0] = 2*(qx*qz - qw*qy); R[:, 2, 1] = 2*(qy*qz + qw*qx); R[:, 2, 2] = 1 - 2*(qx**2 + qy**2)
        return R

    def forward(self, state, action):
        p = state[:, 0:3]; v = state[:, 3:6]; q = state[:, 6:10]
        w_body = state[:, 10:13]; motor_speeds = state[:, 13:17]
        B = state.shape[0]; dev = state.device

        # Nonlinear motor command
        k_cmd = self.motor_k_cmd; u = action
        inner = torch.clamp(k_cmd * u * u + (1 - k_cmd) * u, min=0.0)
        cmd_speeds = (self.omega_max - self.omega_min) * torch.sqrt(inner) + self.omega_min
        motor_dot = (cmd_speeds - motor_speeds) / self.motor_tau
        motor_new = torch.clamp(motor_speeds + motor_dot * self.dt, self.omega_min, self.omega_max)

        if self.prev_motor_speeds is None:
            self.prev_motor_speeds = motor_speeds.clone()
        motor_accel = (motor_new - self.prev_motor_speeds) / self.dt
        self.prev_motor_speeds = motor_new.clone()

        # Body-frame velocity
        R = self._quat_to_rot_matrix(q)
        v_body = torch.bmm(R.transpose(1, 2), v.unsqueeze(-1)).squeeze(-1)
        vx_B, vy_B, vz_B = v_body[:, 0], v_body[:, 1], v_body[:, 2]

        # Aero forces
        omega_sum = motor_new.sum(dim=-1)
        omega_sq_sum = (motor_new**2).sum(dim=-1)
        omega_bar = omega_sum.clamp(min=1.0)
        alpha = torch.atan2(vz_B, PROP_RADIUS * omega_bar)
        v_hor = torch.sqrt(vx_B**2 + vy_B**2 + 1e-8)
        mu = torch.atan2(v_hor, PROP_RADIUS * omega_bar)

        Fx = -self.k_x * vx_B * omega_sum - self.k_x2 * vx_B * vx_B.abs()
        Fy = -self.k_y * vy_B * omega_sum - self.k_y2 * vy_B * vy_B.abs()
        Fz = -self.k_omega * (1 + self.k_alpha * alpha + self.k_hor * mu) * omega_sq_sum
        F_body = torch.stack([Fx, Fy, Fz], dim=-1)

        # Translation
        F_world = torch.bmm(R, F_body.unsqueeze(-1)).squeeze(-1)
        v_dot = F_world + self.g.to(dev).unsqueeze(0)
        v_new = v + v_dot * self.dt
        p_new = p + v_new * self.dt

        # Moments
        w1, w2, w3, w4 = motor_new[:, 0], motor_new[:, 1], motor_new[:, 2], motor_new[:, 3]
        w1sq, w2sq, w3sq, w4sq = w1**2, w2**2, w3**2, w4**2
        kp = self.kp.to(dev); kq = self.kq.to(dev)
        kr_speed = self.kr_speed.to(dev); kr_accel = self.kr_accel.to(dev)
        p_rate, q_rate, r_rate = w_body[:, 0], w_body[:, 1], w_body[:, 2]

        Mx = -kp[0]*w1sq - kp[1]*w2sq + kp[2]*w3sq + kp[3]*w4sq + self.Jx * q_rate * r_rate
        My = -kq[0]*w1sq + kq[1]*w2sq - kq[2]*w3sq + kq[3]*w4sq + self.Jy * p_rate * r_rate
        wd1, wd2, wd3, wd4 = motor_accel[:, 0], motor_accel[:, 1], motor_accel[:, 2], motor_accel[:, 3]
        Mz = (-kr_speed[0]*w1 + kr_speed[1]*w2 + kr_speed[2]*w3 - kr_speed[3]*w4
              - kr_accel[0]*wd1 + kr_accel[1]*wd2 + kr_accel[2]*wd3 - kr_accel[3]*wd4
              + self.Jz * p_rate * q_rate)

        w_dot = torch.stack([Mx, My, Mz], dim=-1)
        w_body_new = w_body + w_dot * self.dt

        # Quaternion integration
        qw, qx, qy, qz = q[:, 0], q[:, 1], q[:, 2], q[:, 3]
        wx, wy, wz = w_body_new[:, 0], w_body_new[:, 1], w_body_new[:, 2]
        q_dot = torch.stack([
            0.5 * (-qx*wx - qy*wy - qz*wz),
            0.5 * (qw*wx + qy*wz - qz*wy),
            0.5 * (qw*wy - qx*wz + qz*wx),
            0.5 * (qw*wz + qx*wy - qy*wx),
        ], dim=-1)
        q_new = q + q_dot * self.dt
        q_new = q_new / torch.linalg.norm(q_new, dim=-1, keepdim=True)

        return torch.cat([p_new, v_new, q_new, w_body_new, motor_new], dim=-1)

print("Dynamics loaded")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# MonoRace Gymnasium Environment (M23 paper-exact)
# ═══════════════════════════════════════════════════════════════════

class MonoRaceSimEnv(gym.Env):
    """24D obs, M23 reward, 11-gate track, full system-identified dynamics."""

    def __init__(self):
        super().__init__()
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(OBS_DIM,), dtype=np.float32)
        self.action_space = spaces.Box(low=0.0, high=1.0, shape=(4,), dtype=np.float32)
        self.dynamics = QuadcopterDynamics(dt=DT_SIM)
        self.gates = GATE_POSITIONS
        self.state = None
        self.prev_state_np = None
        self.step_count = 0
        self.current_gate_idx = 0
        self.max_steps = MAX_EPISODE_STEPS

    def _gate_rotation_inv(self, gate_yaw):
        c, s = np.cos(-gate_yaw), np.sin(-gate_yaw)
        return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])

    def _world_to_gate_frame(self, pos_w, vel_w, euler_w, gate_idx):
        gx, gy, gz, g_yaw = self.gates[gate_idx % NUM_GATES]
        gate_pos = np.array([gx, gy, gz])
        R_inv = self._gate_rotation_inv(g_yaw)
        p_g = R_inv @ (pos_w - gate_pos)
        v_g = R_inv @ vel_w
        euler_g = euler_w.copy()
        euler_g[2] -= g_yaw
        return p_g, v_g, euler_g

    def _next_gate_in_current_frame(self, gate_idx):
        gi = gate_idx % NUM_GATES
        gi_next = (gate_idx + 1) % NUM_GATES
        gx, gy, gz, g_yaw = self.gates[gi]
        gx_n, gy_n, gz_n, g_yaw_n = self.gates[gi_next]
        gate_pos = np.array([gx, gy, gz])
        next_pos = np.array([gx_n, gy_n, gz_n])
        R_inv = self._gate_rotation_inv(g_yaw)
        p_next_g = R_inv @ (next_pos - gate_pos)
        yaw_rel = (g_yaw_n - g_yaw + np.pi) % (2 * np.pi) - np.pi
        return p_next_g, yaw_rel

    def _get_obs(self):
        state_np = self.state.cpu().numpy().squeeze(0)
        p_w, v_w, q = state_np[0:3], state_np[3:6], state_np[6:10]
        w_body, motor_speeds = state_np[10:13], state_np[13:17]
        euler = quat_to_euler(q)

        # World-frame angular velocity
        qw, qx, qy, qz = q
        R = np.array([
            [1-2*(qy**2+qz**2), 2*(qx*qy-qw*qz), 2*(qx*qz+qw*qy)],
            [2*(qx*qy+qw*qz), 1-2*(qx**2+qz**2), 2*(qy*qz-qw*qx)],
            [2*(qx*qz-qw*qy), 2*(qy*qz+qw*qx), 1-2*(qx**2+qy**2)],
        ])
        w_world = R @ w_body

        gi = self.current_gate_idx
        p_g, v_g, euler_g = self._world_to_gate_frame(p_w, v_w, euler, gi)
        p_next_g, yaw_next_rel = self._next_gate_in_current_frame(gi)
        gx, gy, gz, _ = self.gates[gi % NUM_GATES]
        dist_gate = np.linalg.norm(p_w - np.array([gx, gy, gz]))

        return np.concatenate([
            p_g, v_g, euler_g, w_world, w_body,
            motor_speeds / MOTOR_OMEGA_MAX,
            p_next_g, [yaw_next_rel], [dist_gate],
        ]).astype(np.float32)

    def _check_gate_passage(self, p_prev, p_curr, gate_idx):
        gx, gy, gz, g_yaw = self.gates[gate_idx % NUM_GATES]
        gate_pos = np.array([gx, gy, gz])
        normal = np.array([np.cos(g_yaw), np.sin(g_yaw), 0.0])
        d_prev = np.dot(p_prev - gate_pos, normal)
        d_curr = np.dot(p_curr - gate_pos, normal)
        if d_prev * d_curr > 0:
            return False, False, 0.0
        t = d_prev / (d_prev - d_curr + 1e-8)
        p_cross = p_prev + t * (p_curr - p_prev)
        R_inv = self._gate_rotation_inv(g_yaw)
        local = R_inv @ (p_cross - gate_pos)
        half = GATE_SIZE / 2.0
        offset = np.sqrt(local[1]**2 + local[2]**2)
        if abs(local[1]) < half and abs(local[2]) < half:
            return True, False, offset
        else:
            return False, True, offset

    def _compute_reward(self, state_np, prev_state_np, action):
        p, p_prev = state_np[0:3], prev_state_np[0:3]
        w_body, q = state_np[10:13], state_np[6:10]
        gi = self.current_gate_idx % NUM_GATES
        gate_pos = np.array(self.gates[gi][:3])

        dist_prev = np.linalg.norm(p_prev - gate_pos)
        dist_curr = np.linalg.norm(p - gate_pos)
        r_prog = LAMBDA_PROG * min(dist_prev - dist_curr, V_MAX * DT_SIM)

        r_gate = r_offset = 0.0
        passed, hit_frame, offset = self._check_gate_passage(p_prev, p, gi)
        if passed:
            r_gate = LAMBDA_GATE
            r_offset = -LAMBDA_OFFSET * offset
            self.current_gate_idx += 1
        elif hit_frame:
            self._crashed = True

        r_rate = -LAMBDA_RATE * np.sum(w_body**2)

        euler = quat_to_euler(q)
        cam_off = np.sqrt(euler[0]**2 + euler[1]**2)
        r_perc = -LAMBDA_PERC * cam_off if cam_off > (np.pi / 3) else 0.0

        return r_prog + r_gate + r_offset + r_rate + r_perc

    def _find_nearest_gate_ahead(self, pos):
        best_idx, best_dist = 0, np.inf
        for i, (gx, gy, gz, g_yaw) in enumerate(self.gates):
            gate_pos = np.array([gx, gy, gz])
            normal = np.array([np.cos(g_yaw), np.sin(g_yaw), 0.0])
            if np.dot(pos - gate_pos, normal) < 0:
                dist = np.linalg.norm(pos - gate_pos)
                if dist < best_dist:
                    best_dist = dist
                    best_idx = i
        return best_idx

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        dr = lambda nom, rng=DR_RANGE: nom * self.np_random.uniform(1 - rng, 1 + rng)
        dr4 = lambda nom, rng=DR_RANGE: [nom[i] * self.np_random.uniform(1 - rng, 1 + rng) for i in range(4)]
        params = {
            'k_omega': dr(K_OMEGA), 'k_x': dr(K_X), 'k_y': dr(K_Y),
            'k_x2': dr(K_X2), 'k_y2': dr(K_Y2), 'k_alpha': dr(K_ALPHA), 'k_hor': dr(K_HOR),
            'kp': dr4(K_P), 'kq': dr4(K_Q), 'kr_speed': dr4(K_R_SPEED), 'kr_accel': dr4(K_R_ACCEL),
            'Jx': dr(J_X), 'Jy': dr(J_Y), 'Jz': dr(J_Z),
            'motor_tau': dr(MOTOR_TAU, DR_TAU), 'motor_k_cmd': dr(MOTOR_K_CMD),
            'omega_min': dr(MOTOR_OMEGA_MIN), 'omega_max': dr(MOTOR_OMEGA_MAX, DR_OMEGA_MAX),
        }
        self.dynamics = QuadcopterDynamics(dt=DT_SIM, params=params)
        px = self.np_random.uniform(*INIT_X_RANGE)
        py = self.np_random.uniform(*INIT_Y_RANGE)
        pz = self.np_random.uniform(*INIT_Z_RANGE)
        vx, vy, vz = [self.np_random.uniform(*INIT_V_RANGE) for _ in range(3)]
        roll = self.np_random.uniform(*INIT_RP_RANGE)
        pitch = self.np_random.uniform(*INIT_RP_RANGE)
        yaw = self.np_random.uniform(*INIT_YAW_RANGE)
        q = euler_to_quat(roll, pitch, yaw)
        wx, wy, wz = [self.np_random.uniform(*INIT_OMEGA_RANGE) for _ in range(3)]

        self.state = torch.cat([
            torch.tensor([[px, py, pz]], dtype=torch.float32),
            torch.tensor([[vx, vy, vz]], dtype=torch.float32),
            torch.tensor(q, dtype=torch.float32).unsqueeze(0),
            torch.tensor([[wx, wy, wz]], dtype=torch.float32),
            torch.full((1, 4), MOTOR_OMEGA_MIN, dtype=torch.float32),
        ], dim=-1)
        self.prev_state_np = self.state.cpu().numpy().squeeze(0).copy()
        self.step_count = 0
        self._crashed = False
        self.current_gate_idx = self._find_nearest_gate_ahead(np.array([px, py, pz]))
        return self._get_obs(), {}

    def step(self, action):
        self.prev_state_np = self.state.cpu().numpy().squeeze(0).copy()
        action_tensor = torch.tensor(action, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            self.state = self.dynamics(self.state, action_tensor)
        self.step_count += 1
        state_np = self.state.cpu().numpy().squeeze(0)

        reward = self._compute_reward(state_np, self.prev_state_np, action)

        p = state_np[0:3]
        w_body = state_np[10:13]
        out_of_bounds = (p[0] < BOUNDS_X[0] or p[0] > BOUNDS_X[1] or
                         p[1] < BOUNDS_Y[0] or p[1] > BOUNDS_Y[1] or
                         p[2] < BOUNDS_Z[0] or p[2] > BOUNDS_Z[1])
        omega_exceeded = np.linalg.norm(w_body) > OMEGA_MAX_TERMINATION
        crashed = out_of_bounds or self._crashed or omega_exceeded

        if np.any(np.isnan(state_np)):
            crashed = True
            reward = -LAMBDA_CRASH
        if crashed:
            reward -= LAMBDA_CRASH

        completed = self.current_gate_idx >= NUM_GATES * NUM_LAPS
        terminated = crashed or completed
        truncated = self.step_count >= self.max_steps
        obs = np.nan_to_num(self._get_obs(), nan=0.0, posinf=100.0, neginf=-100.0)
        return obs, float(reward), terminated, truncated, {}

# Quick test
env = MonoRaceSimEnv()
obs, _ = env.reset()
obs2, r, *_ = env.step(np.array([0.5, 0.5, 0.5, 0.5]))
print(f"Env OK: obs={obs.shape}, reward={r:.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Training — SB3 PPO (paper-exact M23 config)
# ═══════════════════════════════════════════════════════════════════
# Supports resume: if a checkpoint exists in SAVE_DIR, it auto-resumes.

from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv, VecNormalize
from stable_baselines3.common.callbacks import CheckpointCallback
import glob

def make_env():
    def _init():
        return MonoRaceSimEnv()
    return _init

env = SubprocVecEnv([make_env() for _ in range(NUM_ENVS)])
env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.0)

policy_kwargs = dict(net_arch=[64, 64, 64], activation_fn=nn.ReLU)

# Check for existing checkpoint to resume
checkpoints = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, 'gcnet_m23_*.zip')))
vecnorm_path = os.path.join(SAVE_DIR, 'vecnormalize_m23.pkl')

if checkpoints:
    latest = checkpoints[-1]
    print(f"Resuming from: {latest}")
    if os.path.exists(vecnorm_path):
        env = VecNormalize.load(vecnorm_path, env)
    model = PPO.load(latest, env=env, device="cpu")
    reset_timesteps = False
else:
    print("Starting fresh training")
    model = PPO(
        "MlpPolicy", env,
        policy_kwargs=policy_kwargs,
        learning_rate=3e-4,
        n_steps=2048,
        batch_size=256,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.0,  # M23 paper value
        verbose=1,
        device="cpu",
        tensorboard_log=os.path.join(SAVE_DIR, 'tb_logs'),
    )
    reset_timesteps = True

checkpoint_cb = CheckpointCallback(
    save_freq=500_000 // NUM_ENVS,  # save every 500K steps
    save_path=CHECKPOINT_DIR,
    name_prefix="gcnet_m23",
)

print(f"Training: {TOTAL_TIMESTEPS:,} steps, {NUM_ENVS} envs, dt={DT_SIM}")
model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=checkpoint_cb,
    reset_num_timesteps=reset_timesteps,
)

# Save final model + normalizer to Drive
model.save(os.path.join(SAVE_DIR, 'gcnet_m23_final'))
env.save(vecnorm_path)
print(f"Done! Model saved to {SAVE_DIR}/gcnet_m23_final.zip")